# Cross-Annotation Check: QuranMorph

## Findings

QuranMorph (SinaLab/Birzeit, 2025) is the only machine-readable morphological annotation of
the Quran produced independently of the Quranic Arabic Corpus: three linguists manually
lemmatized all 77,429 words against the Qabas lexicon. Re-examining this project's counts
under it:

1. **The two annotations align word-for-word — verified, not assumed**: the same 77,429
   words, equal word counts in all 6,236 verses, and the word *strings* match
   position-by-position — 77,218 of 77,429 letter skeletons are identical, and the 211
   exceptions are text artifacts of the QuranMorph distribution (208 words carry a stray
   trailing نن in both its `word` and `verse` columns — elided-yāʾ vocatives like رَبِّ —
   and 3 differ by a single orthographic sign). None is a word-boundary disagreement (§1).
2. **50 of 58 word selections agree exactly** (yawm 405, raḥma 114, shaytan 88, īmān 45,
   ṣalāt 83, nās 241, …); the 8 differences are systematic lexicon-design differences —
   seven merges, plus the Akhira feature-scheme case of finding 4 — each examined
   individually in §3 and §5.
3. **The project's audit corrections are independently corroborated**: QuranMorph splits
   *barr* into بَرٌّ "land" = 12 and بَارٌّ "dutiful/righteous" = 10 — the same 12 + 10 split
   as README §2.3; and it files the 13 plural "bones" occurrences under عَظِيمٌ, replicating
   the upstream filing documented in §2.1 (§5).
4. **One flagship claim is annotation-scheme-dependent**: dunya/akhira 115 : 115. QuranMorph
   has no selection that yields 115 for "the hereafter" — it lemmatizes those 115 occurrences
   as آخِرُ (100, mostly tagged noun) plus آخِرَةٌ (15), and its noun آخِرُ also covers "last".
   The 115 exists only through QAC's grammatical-gender feature, which QuranMorph's scheme
   does not expose (§5.1).
5. **Lemma-granularity differences are systematic, not random**: QuranMorph merges suppletive
   plurals and -āt plurals into the base lemma (rijāl under رَجُلٌ → 57, matching this
   project's lemma+variants figure; شَجَرٌ unifies the tree lemmas → 26; ḥasanāt and sayyiʾāt
   fold into ḥasana → 31 and sayyiʾa → 58) and splits where Qabas has distinct entries
   (مَلَكٌ 67 + مَلَاكٌ 21 = 88 for the angels) (§5).
6. **Claim verdicts are stable wherever the claim is expressible** (§4): every lemma-level
   claim that held under QAC and is expressible in QuranMorph's scheme holds there too
   (Adam = Isa 25/25, raḥma 114, shaytan 88, yawm-lemma 405, shahr-lemma 21); every decisive
   failure stays failed (ḥayāt/mawt at 76/53, nās/anbiyāʾ at 241/75). Claims that needed
   QAC-specific features (gender, grammatical number, pronoun suffixes, roots) are simply not
   expressible in QuranMorph and are marked as such, including the 115 : 115 case above.

Caveats: QuranMorph annotates lemma + POS only — no roots, no grammatical number, no
segment-level features — so root-level and convention-level claims cannot be cross-checked.
Agreement between the corpora is corroboration by an independent team, not ground truth; where
they disagree, the disagreement measures how much rests on one team's editorial choices.


## 1. Data, provenance, alignment

`data/quranmorph/` holds the QuranMorph distribution verbatim (obtained from the authors'
download form, 2026-06-11): `quran-dataset.csv` (77,429 rows: location, word, Arabic POS tag,
Qabas lemma id + vocalized lemma), `tagset_translation.csv`, and the authors' license/readme
(CC BY 4.0). Citation: Akra, Hammouda & Jarrar, *QuranMorph: Morphologically Annotated Quranic
Corpus*, arXiv:2506.18148.

Alignment is verified in two steps: (a) both corpora must give every verse the same number of
words, and (b) the actual word at every one of the 77,429 positions must be the same string —
QuranMorph's `word` column vs the token reconstructed from the QAC morphology segments,
compared as letter skeletons (the normalization of `validation/validate_locations.py`).
Without (b), equal counts could mask a compensating split+merge inside a verse.


In [1]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../validation")

import pandas as pd
from src.parser import load_morphology
from src.buckwalter import bw_to_arabic
from validate_locations import skeleton, load_tokens_from_morphology

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

df = load_morphology()
qm = pd.read_csv("../data/quranmorph/quran-dataset.csv")
assert len(qm) == 77429

qm_words = qm.groupby(["surah_number", "verse_number"]).word_position.max()
qac_words = df.groupby(["chapter", "verse"]).word.max()
qm_words.index.names = qac_words.index.names = ["c", "v"]
assert len(qm_words) == len(qac_words) == 6236
assert (qm_words == qac_words).all()

# word-string check at every position (letter skeletons)
qac_tokens = load_tokens_from_morphology()
nn_artifact, other_diff = [], []
for row in qm.itertuples():
    loc = (row.surah_number, row.verse_number, row.word_position)
    exp, act = skeleton(bw_to_arabic(qac_tokens[loc])), skeleton(str(row.word))
    if exp != act:
        (nn_artifact if act.endswith("نن") and act[:-2] == exp else other_diff).append(loc)
assert len(nn_artifact) == 208 and len(other_diff) == 3
print(f"QuranMorph: {len(qm):,} words, {qm.qabas_lemma.nunique():,} distinct Qabas lemmas.")
print(f"All 6,236 verse word-counts match QAC; {len(qm) - 211:,}/77,429 word skeletons "
      f"identical.\n208 QuranMorph words carry a stray trailing نن (elided-ya vocatives; "
      f"present in its verse\ntext too), 3 differ by one orthographic sign "
      f"({', '.join(f'{c}:{v}:{w}' for c, v, w in other_diff)}) — no boundary disagreements.")

QML = qm.set_index(["surah_number", "verse_number", "word_position"])
QM_TOTALS = qm.qabas_lemma.value_counts()


QuranMorph: 77,429 words, 4,628 distinct Qabas lemmas.
All 6,236 verse word-counts match QAC; 77,218/77,429 word skeletons identical.
208 QuranMorph words carry a stray trailing نن (elided-ya vocatives; present in its verse
text too), 3 differ by one orthographic sign (55:1:1, 55:65:1, 42:3:1) — no boundary disagreements.


## 2. Occurrence-aligned correspondence

For a QAC selection (a lemma, optionally gender/POS/number-filtered), take its
chapter:verse:word locations and read off the QuranMorph lemma at each. This sidesteps
orthography entirely. `union total` = the total QuranMorph occurrences (anywhere in the text)
of all QuranMorph lemmas the selection maps onto — if QuranMorph draws the same word boundary,
union total equals the QAC count exactly.

In [2]:
def qac_locs(lemma, gender=None, pos=None, number=None):
    sub = df[df.LEM == lemma]
    if gender:
        sub = sub[sub.PGN.str.contains(gender, na=False)]
    if pos:
        sub = sub[sub.POS == pos]
    if number:
        sub = sub[sub.NUMBER == number]
    return list(zip(sub.chapter, sub.verse, sub.word))


def correspondence(lemma, gender=None, pos=None, number=None):
    locs = qac_locs(lemma, gender, pos, number)
    got = QML.loc[locs].qabas_lemma
    dist = got.value_counts()
    union_total = int(QM_TOTALS[dist.index].sum())
    return dist, union_total, len(locs)


# the 38 words' primary selections (from notebook 03) + claim-audit words (notebook 04)
SELECTIONS = [
    # name, qac lemma, gender filter
    ("Dunya", "d~unoyaA", None), ("Akhira", "A^xir", "F"), ("Malak", "malak", None),
    ("Shaytan", "$ayoTa`n", None), ("Hayat", "Hayaw`p", None), ("Mawt", "mawot", None),
    ("Rajul", "rajul", None), ("Imra'a", "{mora>at", None), ("Shahr", "$ahor", None),
    ("Yawm", "yawom", None), ("Bahr", "baHor", None), ("Barr", "bar~", None),
    ("Jannah", "jan~ap", None), ("Jahannam", "jahan~am", None), ("Harr", "Har~", None),
    ("Bard", "barod", None), ("Zakat", "zakaw`p", None), ("Baraka", "baraka`t", None),
    ("Qaala", "qaAla", None), ("Adhab", "Ea*aAb", None), ("Rahma", "raHomap", None),
    ("Maghfira", "m~agofirap", None), ("Ghani", "ganiY~", None), ("Faqir", "faqiyr", None),
    ("Hasana", "Hasanap", None), ("Sayyi'a", "say~i}ap", None), ("Adam", "A^dam", None),
    ("Isa", "EiysaY", None), ("Insan", "<insa`n", None), ("Iblis", "<iboliys", None),
    ("Iman", "<iyma`n", None), ("Kufr", "kufor", None), ("Turab", "turaAb", None),
    ("Nutfa", "n~uTofap", None), ("Alaqa", "Ealaqap", None), ("Mudgha", "muDogap", None),
    ("Izam", "EaZom", None), ("Lahm", "laHom", None),
    # claims-audit additions
    ("Salihat", "S~a`liHa`t", None), ("Sayyi'at", "say~i_#aAt", None),
    ("Nas", "n~aAs", None), ("Nabiy", "n~abiY~", None), ("Muhammad", "muHam~ad", None),
    ("Sharia", "$ariyEap", None), ("Musiba", "m~uSiybap", None), ("Shukr", "$ukor", None),
    ("Jahr", "jahor", None), ("Alaniya", "EalaAniyap", None), ("Lisan", "lisaAn", None),
    ("Maw'iza", "m~awoEiZap", None), ("Nabat", "nabaAt", None), ("Shajar(a)", "$ajarap", None),
    ("Jaza'", "jazaA^'", None), ("Salat", "Salaw`p", None), ("Naf'", "nafoE", None),
    ("Fasad", "fasaAd", None), ("Yusr", "yusor", None), ("Usr", "Eusor", None),
]

rows = []
for name, lem, gender in SELECTIONS:
    dist, union_total, n = correspondence(lem, gender)
    rows.append({"word": name, "qac_lemma": lem, "qac_count": n,
                 "qm_lemmas": "; ".join(f"{l}={c}" for l, c in dist.items()),
                 "qm_union_total": union_total,
                 "exact_agree": union_total == n})
corr = pd.DataFrame(rows)
display(corr)
corr.to_csv("../output/quranmorph_lemma_map.csv", index=False)
agree = int(corr.exact_agree.sum())
print(f"Saved output/quranmorph_lemma_map.csv — exact agreement (union total == QAC count): "
      f"{agree}/{len(corr)}")

,word,qac_lemma,qac_count,qm_lemmas,qm_union_total,exact_agree
0,Dunya,d~unoyaA,115,دُنْيا=115,115,True
1,Akhira,A^xir,115,آخِرُ=100; آخِرَةٌ=15,155,False
2,Malak,malak,88,مَلَكٌ=67; مَلَاكٌ=21,88,True
3,Shaytan,$ayoTa`n,88,شَيْطَانٌ=88,88,True
4,Hayat,Hayaw`p,76,حَياةٌ=76,76,True
5,Mawt,mawot,50,مَوْتٌ=50,53,False
6,Rajul,rajul,29,رَجُلٌ=29,57,False
7,Imra'a,{mora>at,26,اِمْرَأَةٌ=26,26,True
8,Shahr,$ahor,21,شَهْرٌ=21,21,True
9,Yawm,yawom,405,يَوْمٌ=405,405,True


Saved output/quranmorph_lemma_map.csv — exact agreement (union total == QAC count): 50/58


## 3. Where the union totals differ, and why

A mismatch means the two teams drew word boundaries differently. Every mismatch is listed and
classified — merges (QuranMorph files a suppletive plural or variant under the base lemma),
splits (Qabas has separate entries), and scope differences (a QuranMorph lemma also covers
words outside the QAC selection).

In [3]:
mism = corr[~corr.exact_agree]
display(mism[["word", "qac_count", "qm_lemmas", "qm_union_total"]].reset_index(drop=True))
print(f"{len(mism)} of {len(corr)} selections have a boundary difference; each is examined "
      f"in section 5 or annotated in the stability table.")

,word,qac_count,qm_lemmas,qm_union_total
0,Akhira,115,آخِرُ=100; آخِرَةٌ=15,155
1,Mawt,50,مَوْتٌ=50,53
2,Rajul,29,رَجُلٌ=29,57
3,Hasana,28,حَسَنَةٌ=28,31
4,Sayyi'a,22,سَيِّئةٌ=22,58
5,Salihat,62,صَالِحٌ3=62,127
6,Sayyi'at,36,سَيِّئةٌ=36,58
7,Shajar(a),19,شَجَرٌ=19,26


8 of 58 selections have a boundary difference; each is examined in section 5 or annotated in the stability table.


## 4. Claim-verdict stability

Every lemma-level-expressible claim recounted under QuranMorph lemmas. `n/a (features)` =
the claim's counting needs gender / number / pronoun-suffix / root features that QuranMorph
does not annotate, so it cannot be expressed there at all.

In [4]:
def qm_lemmas_of(*qac_lemmas, gender=None, pos=None, number=None):
    """The set of QuranMorph lemmas a QAC selection maps onto (derived, never typed)."""
    out = []
    for lem in qac_lemmas:
        dist = QML.loc[qac_locs(lem, gender, pos, number)].qabas_lemma.value_counts()
        out.extend(dist.index.tolist())
    return list(dict.fromkeys(out))


def qm_count(*qac_lemmas, **kw):
    """Total QuranMorph occurrences of all QM lemmas the QAC selection maps onto."""
    return int(QM_TOTALS[qm_lemmas_of(*qac_lemmas, **kw)].sum())


AKHIRA_QM = qm_lemmas_of("A^xir")  # derived strings for the akhira discussion

STABILITY = [
    ("C01", "Dunya = Akhira", "115:115", "holds (lemma, gender filter)",
     f"dunya {qm_count('d~unoyaA')}; akhira: no 115 selection — its 115 words map to "
     f"{dict(QML.loc[qac_locs('A^xir', gender='F')].qabas_lemma.value_counts())}, and the big lemma also covers 'last'",
     "SCHEME-DEPENDENT"),
    ("C02", "Mala'ika = Shayatin", "88:88", "holds (lemma)",
     f"shaytan {qm_count('$ayoTa`n')}; malak (2-lemma union) {qm_count('malak')}",
     "stable (via 2-lemma union)"),
    ("C03", "Hayat = Mawt", "145:145", "does not hold",
     f"{qm_count('Hayaw`p')} vs {int(QM_TOTALS[qm_lemmas_of('mawot')[0]])} — nowhere near 145 either",
     "stable (fails)"),
    ("C05", "Salihat = Sayyi'at", "167:167", "one side only (root)",
     f"lemma level {qm_count('S~a`liHa`t')} vs {qm_count('say~i_#aAt')}; root level not expressible",
     "consistent (lemma); root n/a"),
    ("C11", "Naf' = Fasad", "50:50", "holds (root)",
     f"root not expressible; lemma level {qm_count('nafoE')} = {qm_count('fasaAd')} (an equality QuranMorph also shows)",
     "root n/a; lemma equality replicates"),
    ("C12", "Nas = Anbiya'", "50:50", "does not hold",
     f"{qm_count('n~aAs')} vs {qm_count('n~abiY~')}",
     "stable (fails)"),
    ("C13", "Muhammad = Sharia", "4:4", "one side only",
     f"{qm_count('muHam~ad')} vs {qm_count('$ariyEap')}",
     "stable (one side)"),
    ("C16", "Jahr = 'Alaniya", "16:16", "holds (root)",
     f"root not expressible; lemmas {qm_count('jahor')} vs {qm_count('EalaAniyap')}",
     "n/a (root)"),
    ("C18", "Lisan = Maw'iza", "25:25", "holds (root)",
     f"root not expressible; lisan lemma {qm_count('lisaAn')} agrees with QAC's 25",
     "n/a (root); lisan side stable"),
    ("C19", "Nabat = Shajar", "26:26", "mixed methods only",
     f"{qm_count('nabaAt')} vs {qm_count('$ajar')} — shajar = 26 is a single lemma here, nabat is still 9",
     "stable (one side per method)"),
    ("C20", "Jaza' : Maghfira = 117:234", "—", "one side only (root)",
     f"root not expressible; lemmas {qm_count('jazaA^' + chr(39))} vs {qm_count('m~agofirap')} agree with QAC",
     "n/a (root)"),
    ("C21", "Adam = Isa", "25:25", "holds (lemma)",
     f"{qm_count('A^dam')} = {qm_count('EiysaY')}",
     "STABLE (holds)"),
    ("C22", "Yawm = 365", "365", "holds (singular, no pron suffix)",
     f"lemma {qm_count('yawom')} agrees with QAC's 405; the 365 convention needs number+suffix features",
     "n/a (features); lemma stable"),
    ("C24", "Shahr = 12", "12", "holds (singular)",
     f"lemma {qm_count('$ahor')} agrees with QAC's 21; singular not expressible",
     "n/a (features); lemma stable"),
    ("C25", "Bahr : Barr = 32:13", "—", "holds (singular definite)",
     f"definite/singular not expressible; lemmas {qm_count('baHor')} vs barr-split "
     f"{dict(QML.loc[qac_locs('bar~')].qabas_lemma.value_counts())} — the 12 independently confirms the land count (README 2.3)",
     "n/a (features); land=12 corroborated"),
    ("C28", "Rahma = 114", "114", "holds (lemma)",
     f"{qm_count('raHomap')}",
     "STABLE (holds)"),
]
stab = pd.DataFrame(STABILITY, columns=["id", "claim", "claimed", "qac_verdict",
                                        "quranmorph", "stability"])
display(stab)
stab.to_csv("../output/quranmorph_crosscheck.csv", index=False)
print("Saved output/quranmorph_crosscheck.csv")

,id,claim,claimed,qac_verdict,quranmorph,stability
0,C01,Dunya = Akhira,115:115,"holds (lemma, gender filter)","dunya 115; akhira: no 115 selection — its 115 words map to {'آخِرُ': np.int64(100), 'آ...",SCHEME-DEPENDENT
1,C02,Mala'ika = Shayatin,88:88,holds (lemma),shaytan 88; malak (2-lemma union) 88,stable (via 2-lemma union)
2,C03,Hayat = Mawt,145:145,does not hold,76 vs 53 — nowhere near 145 either,stable (fails)
3,C05,Salihat = Sayyi'at,167:167,one side only (root),lemma level 127 vs 58; root level not expressible,consistent (lemma); root n/a
4,C11,Naf' = Fasad,50:50,holds (root),root not expressible; lemma level 11 = 11 (an equality QuranMorph also shows),root n/a; lemma equality replicates
5,C12,Nas = Anbiya',50:50,does not hold,241 vs 75,stable (fails)
6,C13,Muhammad = Sharia,4:4,one side only,4 vs 1,stable (one side)
7,C16,Jahr = 'Alaniya,16:16,holds (root),root not expressible; lemmas 7 vs 4,n/a (root)
8,C18,Lisan = Maw'iza,25:25,holds (root),root not expressible; lisan lemma 25 agrees with QAC's 25,n/a (root); lisan side stable
9,C19,Nabat = Shajar,26:26,mixed methods only,"9 vs 26 — shajar = 26 is a single lemma here, nabat is still 9",stable (one side per method)


Saved output/quranmorph_crosscheck.csv


## 5. Divergence forensics

### 5.1 Akhira — the flagship pair is annotation-scheme-dependent

QAC represents "the hereafter" as the feminine slice (115) of lemma `A^xir` (155 = 115 F +
40 M "last/latter", README §2.2). QuranMorph, which has no gender feature, lemmatizes the same
155 words as آخِرُ vs آخِرَةٌ — and draws the line elsewhere:

In [5]:
ax = df[df.LEM == "A^xir"]
fem = ax[ax.PGN.str.contains("F", na=False)]
masc = ax[~ax.PGN.str.contains("F", na=False)]
for name, sub in [("QAC feminine = 'the hereafter' (115)", fem),
                  ("QAC masculine = 'last/latter' (40)", masc)]:
    locs = list(zip(sub.chapter, sub.verse, sub.word))
    got = QML.loc[locs]
    print(name)
    print(got.groupby(["qabas_lemma", "POS"]).size().to_string(), "\n")
print("QuranMorph totals:")
print(qm[qm.qabas_lemma.isin(["آخِرُ", "آخِرَةٌ"])].groupby(["qabas_lemma", "POS"]).size().to_string())
print(f"""
No QuranMorph selection yields 115: آخِرَةٌ = 15; آخِرُ as noun = 119 (97 'hereafter' + 22
'last'). The 115 : 115 equality is countable only through QAC's gender feature. dunya = 115 is
identical in both. This does not make the claim false — it makes it dependent on one
annotation scheme's feature set, which is exactly what a cross-check is for.""")

QAC feminine = 'the hereafter' (115)
qabas_lemma  POS
آخِرَةٌ      اسم    15
آخِرُ        اسم    97
             صفة     3 

QAC masculine = 'last/latter' (40)
qabas_lemma  POS
آخِرُ        اسم    22
             صفة    17
             ظرف     1 

QuranMorph totals:
qabas_lemma  POS
آخِرَةٌ      اسم     15
آخِرُ        اسم    119
             صفة     20
             ظرف      1

No QuranMorph selection yields 115: آخِرَةٌ = 15; آخِرُ as noun = 119 (97 'hereafter' + 22
'last'). The 115 : 115 equality is countable only through QAC's gender feature. dunya = 115 is
identical in both. This does not make the claim false — it makes it dependent on one
annotation scheme's feature set, which is exactly what a cross-check is for.


### 5.2 Corroborations of the audit corrections

In [6]:
# barr: QuranMorph independently splits land (12) from dutiful/righteous (10) — README 2.3
locs = qac_locs("bar~")
print("QAC bar~ (22) under QuranMorph:")
print(QML.loc[locs].qabas_lemma.value_counts().to_string())

# bones: QuranMorph files the 13 plural occurrences under 'aZiym' (great) as well — README 2.1
bones = QML.loc[qac_locs("EaZiym", pos="N", number="P")]
print("\nthe 13 plural 'bones' rows (QAC: LEM EaZiym, N, P) under QuranMorph:")
print(bones.qabas_lemma.value_counts().to_string())
print("\nQuranMorph also files عظام under the 'great' lemma — the same upstream-style filing")
print(f"(its dedicated bone lemma عَظْمٌ totals {int(QM_TOTALS['عَظْمٌ'])}, the two singulars).")
print("Two independent teams made the same call; counting 'bones' requires deliberate selection in both.")

QAC bar~ (22) under QuranMorph:
qabas_lemma
بَرٌّ     12
بَارٌّ    10

the 13 plural 'bones' rows (QAC: LEM EaZiym, N, P) under QuranMorph:
qabas_lemma
عَظِيمٌ    13

QuranMorph also files عظام under the 'great' lemma — the same upstream-style filing
(its dedicated bone lemma عَظْمٌ totals 2, the two singulars).
Two independent teams made the same call; counting 'bones' requires deliberate selection in both.


### 5.3 Merges: where QuranMorph's lemma equals QAC's lemma+variants

QuranMorph files suppletive plurals and same-meaning variants under the base entry — landing
on the figures this project reaches via declared variant selectors (notebook 03 §3) — plus,
for shajara + shajar, the two-lemma claim side of notebook 04 (C19):

In [7]:
checks = [
    ("Rajul + rijal", ["rajul", "rijaAl"]),
    ("Mawt + mawtat/mamat", ["mawot", "mawotat", "mamaAt"]),
    ("Shajara + shajar", ["$ajarap", "$ajar"]),
    ("Hasana + hasanat", ["Hasanap", "Hasana`t"]),
    ("Sayyi'a + sayyi'at", ["say~i}ap", "say~i_#aAt"]),
]
for name, qac_lems in checks:
    qac_n = int(df.LEM.isin(qac_lems).sum())
    sub = df[df.LEM.isin(qac_lems)]
    locs = list(zip(sub.chapter, sub.verse, sub.word))
    dist = QML.loc[locs].qabas_lemma.value_counts()
    union = int(QM_TOTALS[dist.index].sum())
    print(f"{name}: QAC {qac_n}  ->  QM {dict(dist)}  (union total {union})")
print("""
QuranMorph's rajul lemma = 57 equals this project's rajul lemma+variants (29 + 28 rijal);
its shajar lemma = 26 merges the two tree lemmas, and hasana = 31 / sayyi'a = 58 fold the
separate -at plural lemmas into the base word. Where notebook 03 declared variants by hand,
QuranMorph reaches the same totals by lexicon design — independent support for the variant
selections.""")


Rajul + rijal: QAC 57  ->  QM {'رَجُلٌ': np.int64(57)}  (union total 57)


Mawt + mawtat/mamat: QAC 56  ->  QM {'مَوْتٌ': np.int64(53), 'مَمَاتٌ': np.int64(3)}  (union total 56)
Shajara + shajar: QAC 26  ->  QM {'شَجَرٌ': np.int64(26)}  (union total 26)
Hasana + hasanat: QAC 31  ->  QM {'حَسَنَةٌ': np.int64(31)}  (union total 31)


Sayyi'a + sayyi'at: QAC 58  ->  QM {'سَيِّئةٌ': np.int64(58)}  (union total 58)

QuranMorph's rajul lemma = 57 equals this project's rajul lemma+variants (29 + 28 rijal);
its shajar lemma = 26 merges the two tree lemmas, and hasana = 31 / sayyi'a = 58 fold the
separate -at plural lemmas into the base word. Where notebook 03 declared variants by hand,
QuranMorph reaches the same totals by lexicon design — independent support for the variant
selections.


## 6. Sanity checks

Pinned invariants, mirrored in `tests/test_quranmorph.py`.

In [8]:
assert qm_count("$ayoTa`n") == 88
assert qm_count("raHomap") == 114
assert qm_count("yawom") == 405
assert qm_count("A^dam") == 25 and qm_count("EiysaY") == 25
assert qm_count("malak") == 88            # 2-lemma union
akhira_dist = QML.loc[qac_locs("A^xir", gender="F")].qabas_lemma.value_counts()
assert sorted(akhira_dist.values.tolist(), reverse=True) == [100, 15]
assert qm_count("A^xir") == 155           # union covers hereafter+last, != 115
barr_dist = QML.loc[qac_locs("bar~")].qabas_lemma.value_counts()
assert sorted(barr_dist.values.tolist(), reverse=True) == [12, 10]
bones = QML.loc[qac_locs("EaZiym", pos="N", number="P")].qabas_lemma
assert bones.nunique() == 1 and len(bones) == 13
assert int(QM_TOTALS[qm_lemmas_of("rajul")[0]]) == 57
assert qm_count("nafoE") == 11 and qm_count("fasaAd") == 11
print("All QuranMorph cross-check pins verified.")

All QuranMorph cross-check pins verified.
